# PARTE 3 de la guia - Entrenamiento de una Red Neuronal (MLP) usando MNIST

*Clasificación de Dígitos usando MNIST*

En esta guía se implementa y entrena una red neuronal tipo MLP utilizando el dataset MNIST.

El objetivo es clasificar imágenes de números escritos a mano desde 0 hasta 9.

Las clases corresponden a los dígitos:
- 0
- 1
- 2
- 3
- 4
- 5
- 6
- 7
- 8
- 9

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

*1. Configuración*

Se verifica si existe una GPU disponible para acelerar el entrenamiento de la red neuronal. En caso contrario se utilizará CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)

*2. Transformaciones, carga y preparacion de imágenes*

Las imagenes del dataset MNIST se convierten a tensores y se normalizan para mejorar el entrenamiento.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = datasets.MNIST(
    'MNIST_data/',
    download=True,
    train=True,
    transform=transform
)

testset = datasets.MNIST(
    'MNIST_data/',
    download=True,
    train=False,
    transform=transform
)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64, shuffle=True)

print("Cantidad entrenamiento:", len(trainset))
print("Cantidad prueba:", len(testset))

*3. Visualización de ejemplos*

Se va a mostrar algunas imágenes del dataset junto con sus etiquetas reales.

In [ ]:
images, labels = next(iter(trainloader))

fig, axes = plt.subplots(1,5, figsize=(15,5))

for i in range(5):

    img = images[i].squeeze()

    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis("off")

plt.show()

*4. Construcción de la red neuronal*

Se implemento un perceptrón multicapa (MLP) con varias capas ocultas y función de activación ReLU de la siguiente forma usando el dataset MNIST:

In [ ]:
class MNISTModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Flatten(),

            nn.Linear(28 * 28, 400),
            nn.ReLU(),

            nn.Linear(400, 200),
            nn.ReLU(),

            nn.Linear(200, 100),
            nn.ReLU(),

            nn.Linear(100, 10)
        )

    def forward(self, x):
        return self.network(x)

model = MNISTModel().to(device)

print(model)

*5. Función de pérdida y optimizador*

Se utiliza *CrossEntropyLoss* para clasificación y *Adam* como optimizador.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

*6. Entrenamiento de la red neuronal*

En esta etapa la red aprende y logra reconocer los números del dataset MNIST.

In [ ]:
epochs = 10 

train_losses = []
accuracies = []

for epoch in range(epochs):

    running_loss = 0

    model.train()

    for images, labels in trainloader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(trainloader)

    train_losses.append(avg_loss)

    # EVALUACIÓN

    correct = 0
    total = 0

    model.eval()

    with torch.no_grad():

        for images, labels in testloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    accuracies.append(accuracy)

    print(f"Epoch [{epoch+1}/{epochs}]")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.2f}%")
    print("-" * 40)

*7. Grafica de perdida*

Se muestra la evolución de la perdida durante el entrenamiento.

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(train_losses)

plt.title("Loss de entrenamiento")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.show()

*8. Grafica de precision*

Se visualiza la precision obtenida en cada epoch, y finalmente se gurda el modelo entrenado.

In [ ]:
plt.figure(figsize=(10,4))

plt.plot(accuracies)

plt.title("Grafica de Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.show()

torch.save(model.state_dict(), "mnist_model.pth")

print("Modelo guardado correctamente")

*9. Prediccion sobre imagenes*

Se prueba el modelo utilizando imagenes del conjunto de prueba.

In [ ]:
images, labels = next(iter(testloader))

model.eval()

with torch.no_grad():

    outputs = model(images.to(device))

    _, predicted = torch.max(outputs, 1)

plt.imshow(images[0].squeeze(), cmap='gray')

plt.title(f"Prediccion: {predicted[0].item()}")

plt.show()

Con esto finalmente concluimos la  PARTE 3 de la guia - Entrenamiento de una Red Neuronal (MLP) usando MNIST, en donde se implemento una red neuronal tipo MLP utilizando PyTorch, el modelo fue entrenado usando el dataset MNIST, tambien utilizamos capas densas y funciones de activación ReLU, se uso Adam que fue  utilizado como optimizador, se logro que el modelo clasificara imagenes de numeros escritos a mano y see realizaron graficas de perdida y precision para evaluar el entrenamiento.
 